In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import sounddevice as sd
import soundfile
import IPython.display as ipd

from spectralAnalysis import *

In [ ]:
font = {"size": 23}
plt.rc("font", **font)
plt.rc("text", usetex=True)

### Audio record

In [ ]:
def listInputDevices():
    devices = sd.query_devices()
    for idx, device in enumerate(devices):
        if device["max_input_channels"] > 0:
            print(f"{idx}: {device['name']} | nChannels: {device['max_input_channels']} | Sampling rate: {device['default_samplerate']}")

def recordAudio(fileName, duration=5, samplingRate=44100, nChannels=2, dtype="float32", device=None):
    filePath = "./audio/" + str(fileName) + ".wav"

    deviceInfo = sd.query_devices(device, "input")
    maxInputChannels = deviceInfo["max_input_channels"]
    if maxInputChannels <= 0:
        raise ValueError(f"No input channels available")

    if nChannels > maxInputChannels:
        nChannels = maxInputChannels
        print(f"Using {nChannels} channel" + ("s" if nChannels > 1 else "") + " based on device capability.")

    recordedSamples = []
    def callback(input, frames, time, status):
        if status:
            print(status)
        recordedSamples.append(input.copy())
    print("Recording...")

    with sd.InputStream(
        samplerate=samplingRate,
        channels=nChannels,
        dtype=dtype,
        device=device,
        callback=callback
    ):
        while True:
            cmd = input()
            if cmd.strip().lower() == "q": break

    record = np.concatenate(recordedSamples, axis=0)
    soundfile.write(filePath, record, samplingRate)
    return filePath

listInputDevices()

In [ ]:
recordAudio("tmp", samplingRate=44100, nChannels=1, device=1)

### Audio read-in

In [ ]:
def loadAudio(fileName, samplingRate=None, dtype="float32"):
    x, initSamplingRate = soundfile.read(fileName, dtype=dtype)

    if x.ndim != 2 or x.shape[1] != 2:
        if x.ndim == 1:
            x = np.repeat(x[:, None], 2, axis=1)
        else:
            raise ValueError(f"Invalid audio shape {x.shape}")


    if samplingRate is not None:
        nSamples = round(len(x) * samplingRate / initSamplingRate)
        x = sp.signal.resample(x, nSamples)
    else: samplingRate = initSamplingRate

    left, right = x[:, 0], x[:, 1]
    return x, samplingRate, left, right

In [ ]:
file = "./audio/tmp.wav"
x, samplingRate, left, right = loadAudio(file)
ipd.display(ipd.Audio(x.T, rate=samplingRate))

In [ ]:
time = np.linspace(0, len(x) / samplingRate, num=len(x))
plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.fill_between(time, -np.abs(left), np.abs(left), color="tab:blue")
plt.xlabel("Time")
plt.subplot(122)
plt.title("Right")
plt.fill_between(time, -np.abs(right), np.abs(right), color="tab:blue")
plt.xlabel("Time");

### PSD

In [ ]:
correlogramLeft, frequencies = getCorrelogram(left, samplingRate)
correlogramRight, _ = getCorrelogram(right, samplingRate)

plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.plot(frequencies, correlogramLeft)
plt.xlim(0, 1000)
plt.subplot(122)
plt.title("Right")
plt.plot(frequencies, correlogramRight)
plt.xlim(0, 1000);

In [ ]:
periodogramLeft, frequencies = getPeriodogram(left, samplingRate)
periodogramRight, _ = getPeriodogram(right, samplingRate)

plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.plot(frequencies, periodogramLeft)
plt.xlim(0, 1000)
plt.subplot(122)
plt.title("Right")
plt.plot(frequencies, periodogramRight)
plt.xlim(0, 1000);

##### Blackman-Tukey

In [ ]:
psdBlackmanTukeyLeft, frequencies = getBlackmanTukey(left, samplingRate, windowType="hann")
psdBlackmanTukeyRight, _ = getBlackmanTukey(right, samplingRate, windowType="hann")

plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.plot(frequencies, psdBlackmanTukeyLeft)
plt.xlim(0, 1000)
plt.subplot(122)
plt.title("Right")
plt.plot(frequencies, psdBlackmanTukeyRight)
plt.xlim(0, 1000);

##### Bartlett

In [ ]:
psdBartlettLeft, frequencies = getBartlett(left, samplingRate)
psdBartlettRight, _ = getBartlett(right, samplingRate)

plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.plot(frequencies, psdBartlettLeft)
plt.xlim(0, 1000)
plt.subplot(122)
plt.title("Right")
plt.plot(frequencies, psdBartlettRight)
plt.xlim(0, 1000);

##### Welch

In [ ]:
psdWelchLeft, frequencies = getWelch(left, samplingRate, windowType="hann", overlap=0.5)
psdWelchRight, _ = getWelch(right, samplingRate, windowType="hann", overlap=0.5)

plt.figure(figsize=(20, 5))
plt.subplot(121)
plt.title("Left")
plt.plot(frequencies, psdWelchLeft)
plt.xlim(0, 1000)
plt.subplot(122)
plt.title("Right")
plt.plot(frequencies, psdWelchRight)
plt.xlim(0, 1000);

### STFT

In [ ]:
spectrogramLeft, times, frequencies = getSpectrogram(left, samplingRate, 1024, windowType="hann")
spectrogramRight, times, frequencies = getSpectrogram(right, samplingRate, 1024, windowType="hann")

plt.figure(figsize=(20, 20))
plt.subplot(211)
plt.title("Left")
plt.pcolormesh(times, frequencies, spectrogramLeft, vmin=np.max(spectrogramLeft) - 100, cmap="turbo", shading=None)
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Magnitude (dB)")
plt.subplot(212)
plt.title("Right")
plt.pcolormesh(times, frequencies, spectrogramRight, vmin=np.max(spectrogramRight) - 100, cmap="turbo", shading=None)
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Magnitude (dB)");

### Cross-correlation

In [ ]:
rightDelayed = addDelay(right, samplingRate, delay=2)
crossCorrelation, offset = getCCF(left, rightDelayed, samplingRate)
timeDifference = offset[np.argmax(crossCorrelation)]
print(timeDifference)

plt.figure(figsize=(15, 5))
plt.plot(offset, crossCorrelation);

##### Generalized Cross-correlation

In [ ]:
generalizedCrossCorrelation, offset = getGCC(left, rightDelayed, samplingRate)
timeDifference = offset[np.argmax(generalizedCrossCorrelation)]
print(timeDifference)

plt.figure(figsize=(15, 5))
plt.plot(offset, generalizedCrossCorrelation);